In [1]:
import io
import psycopg2
import pandas as pd

path = r'c:\Users\H59045\Downloads\2001Q1\2001Q1.csv'
reader = pd.read_csv(path, sep='|',header=None,dtype=str,chunksize=200_000, usecols=range(50))

conn = psycopg2.connect(dbname='postgres', user='postgres', password=input(), host='localhost')
cur = conn.cursor()

chunk = next(reader)

buf = io.StringIO()
chunk.to_csv(buf, sep='|', header=False, index=False)
buf.seek(0)

cur.copy_expert(
    "COPY fnma_sf_loan_performance FROM STDIN WITH (FORMAT csv, DELIMITER '|')",
    buf
)
conn.commit()
print(len(chunk), "rows loaded")


200000 rows loaded


In [2]:
print(chunk.shape)

(200000, 50)


In [3]:
sql = "COPY fnma_sf_loan_performance FROM STDIN WITH (FORMAT csv, DELIMITER '|')"
loaded = len(chunk)  # the 200k already in

for i, chunk in enumerate(reader, start=1):
    buf = io.StringIO()
    chunk.to_csv(buf, sep='|', header=False, index=False)
    buf.seek(0)
    try:
        cur.copy_expert(sql, buf)
        conn.commit()
    except Exception as e:
        conn.rollback()
        print(f"Failed on chunk {i}: {e}")
        break
    loaded += len(chunk)
    print(f"Chunk {i} done: {loaded:,} rows loaded")

cur.close()
conn.close()


Chunk 1 done: 400,000 rows loaded
Chunk 2 done: 600,000 rows loaded
Chunk 3 done: 800,000 rows loaded
Chunk 4 done: 1,000,000 rows loaded
Chunk 5 done: 1,200,000 rows loaded
Chunk 6 done: 1,400,000 rows loaded
Chunk 7 done: 1,600,000 rows loaded
Chunk 8 done: 1,800,000 rows loaded
Chunk 9 done: 2,000,000 rows loaded
Chunk 10 done: 2,200,000 rows loaded
Chunk 11 done: 2,400,000 rows loaded
Chunk 12 done: 2,600,000 rows loaded
Chunk 13 done: 2,800,000 rows loaded
Chunk 14 done: 3,000,000 rows loaded
Chunk 15 done: 3,200,000 rows loaded
Chunk 16 done: 3,400,000 rows loaded
Chunk 17 done: 3,600,000 rows loaded
Chunk 18 done: 3,800,000 rows loaded
Chunk 19 done: 4,000,000 rows loaded
Chunk 20 done: 4,200,000 rows loaded
Chunk 21 done: 4,400,000 rows loaded
Chunk 22 done: 4,600,000 rows loaded
Chunk 23 done: 4,800,000 rows loaded
Chunk 24 done: 5,000,000 rows loaded
Chunk 25 done: 5,200,000 rows loaded
Chunk 26 done: 5,400,000 rows loaded
Chunk 27 done: 5,600,000 rows loaded
Chunk 28 done: 5